# Vedic Transformer Training
## 133M parameter model — 10 Vedic-Upanishadic-Puranic operations
### Run on free T4 GPU (Runtime → Change runtime type → T4 GPU)

In [ ]:
!git clone https://github.com/divineearthly/sovereign-edge-ai
%cd sovereign-edge-ai
!pip install torch transformers datasets accelerate -q

In [ ]:
import torch
from vedic_transformer import VedicTransformer
from transformers import AutoTokenizer
from datasets import load_dataset
import os

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Create 133M Vedic Transformer
model = VedicTransformer(
    vocab_size=32000,
    dim=1024,
    num_layers=24,
    num_heads=16,
    dropout=0.1
)
model = model.cuda()
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Load Indic text dataset
dataset = load_dataset("ai4bharat/indic-corp", split="train", streaming=True)
dataset = dataset.take(100000)  # 100K examples for initial training
print("Dataset loaded: ai4bharat/indic-corp")

In [ ]:
# Tokenizer (use existing Indic tokenizer as base)
tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

from torch.utils.data import DataLoader
loader = DataLoader(tokenized_dataset, batch_size=4, shuffle=True)
print("DataLoader ready")

In [ ]:
# Training
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
from tqdm import tqdm

optimizer = AdamW(model.parameters(), lr=3e-4)
criterion = CrossEntropyLoss()

model.train()
for epoch in range(3):
    total_loss = 0
    progress = tqdm(loader, desc=f"Epoch {epoch}")
    for batch in progress:
        input_ids = batch["input_ids"].cuda()
        
        logits = model(input_ids)
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = input_ids[:, 1:].contiguous()
        
        loss = criterion(
            shift_logits.view(-1, 32000),
            shift_labels.view(-1)
        )
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        progress.set_postfix(loss=loss.item())
    
    print(f"Epoch {epoch} avg loss: {total_loss/len(loader):.4f}")
    torch.save(model.state_dict(), f"vedic_133m_epoch{epoch}.pt")

In [ ]:
# Save final model
torch.save(model.state_dict(), "vedic_133m_final.pt")

# Upload to HuggingFace
from huggingface_hub import HfApi
api = HfApi()
api.upload_file("vedic_133m_final.pt", "vedic_133m_final.pt", 
                repo_id="divinesouljoy/Vedic-SLM-133M", repo_type="model")
print("Uploaded to HuggingFace: divinesouljoy/Vedic-SLM-133M")